# Selgis ML: проверка бага памяти в ChunkedCrossEntropyLoss

Этот ноутбук проверяет два утверждения из аудита фреймворка **Selgis ML 2.0**:

1. **Старый `ChunkedCrossEntropyLoss` (до фикса) был математически неверен** при `label_smoothing > 0` и молча съедал **+2 байта на каждый логит** VRAM (все чанки exp в fp32 удерживались в графе backward).
2. **Исправленная версия** совпадает с `F.cross_entropy` по значению и градиентам и не добавляет лишней памяти. На LoRA-обучении GPT-2 это разница между OOM и спокойным обучением.

**Как запускать:** Runtime → Change runtime type → **GPU (T4)**, затем «Выполнить всё». Занимает ~5–7 минут.


In [ ]:
# 1. Окружение
import subprocess, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ GPU — включи T4!")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

# Colab ships an old torchao that the current peft refuses to work with
# (peft raises ImportError on torchao < 0.16). torchao is optional for PEFT,
# so removing it is the clean fix. Must run BEFORE importing peft.
subprocess.run(["pip", "uninstall", "-y", "torchao"], capture_output=True)
subprocess.run(["pip", "install", "-q", "peft"], check=True)

import peft, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

## 2. Обе реализации лосса — встроены в ноутбук

`OldChunkedCE` — точная копия реализации из Selgis ML 2.0 **до** фикса.
`FixedChunkedCE` — копия **после** фикса: чанки считаются внутри `torch.utils.checkpoint`,
ничего чанкового не удерживается в графе, label smoothing по формуле PyTorch.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

class OldChunkedCE(nn.Module):
    """Реализация ДО фикса: чанки exp в fp32 остаются в графе backward. """
    def __init__(self, chunk_size=1024, label_smoothing=0.0):
        super().__init__()
        self.chunk_size = chunk_size
        self.label_smoothing = label_smoothing

    def _log_z(self, logits, V):
        gm = torch.full(logits.shape[:2], float("-inf"), device=logits.device, dtype=torch.float32)
        for s in range(0, V, self.chunk_size):
            e = min(s + self.chunk_size, V)
            gm = torch.maximum(gm, logits[:, :, s:e].max(-1).values.float())
        gm = gm.detach()
        se = torch.zeros_like(gm)
        for s in range(0, V, self.chunk_size):
            e = min(s + self.chunk_size, V)
            chunk = logits[:, :, s:e].float() - gm.unsqueeze(-1)
            se = se + chunk.exp().sum(-1)          # <-- весь [B,T,V] fp32 живёт до backward
        return gm + se.log()

    def forward(self, logits, targets):
        V = logits.shape[-1]
        lz = self._log_z(logits, V)
        tl = logits.gather(-1, targets.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        hard = lz - tl
        if self.label_smoothing > 0:               # <-- баг: eps*log(V) — константа без градиента
            hard = (1 - self.label_smoothing) * hard + self.label_smoothing * math.log(V)
        return hard.mean()


class FixedChunkedCE(nn.Module):
    """Реализация ПОСЛЕ фикса: чанки через checkpoint, корректный label smoothing. """
    def __init__(self, chunk_size=1024, label_smoothing=0.0):
        super().__init__()
        self.chunk_size = chunk_size
        self.label_smoothing = label_smoothing

    def _bounds(self, V):
        for s in range(0, V, self.chunk_size):
            yield s, min(s + self.chunk_size, V)

    def _ckpt(self, fn, chunk):
        if torch.is_grad_enabled() and chunk.requires_grad:
            return checkpoint(fn, chunk, use_reentrant=False)
        return fn(chunk)

    def forward(self, logits, targets):
        V = logits.shape[-1]
        gm = torch.full(logits.shape[:2], float("-inf"), device=logits.device, dtype=torch.float32)
        with torch.no_grad():
            for s, e in self._bounds(V):
                gm = torch.maximum(gm, logits[:, :, s:e].max(-1).values.float())
        gm = gm.detach()

        se, sx = None, None
        for s, e in self._bounds(V):
            chunk = logits[:, :, s:e]
            p1 = self._ckpt(lambda c: (c.float() - gm.unsqueeze(-1)).exp().sum(-1), chunk)
            se = p1 if se is None else se + p1
            if self.label_smoothing > 0:
                p2 = self._ckpt(lambda c: c.float().sum(-1), chunk)
                sx = p2 if sx is None else sx + p2
        lz = gm + se.log()

        res = torch.zeros(targets.shape, device=logits.device, dtype=logits.dtype)
        for s, e in self._bounds(V):
            in_chunk = (targets >= s) & (targets < e)
            if not in_chunk.any():
                continue
            def fn(c, s=s):                        # фикс late-binding замыкания
                idx = (targets - s).clamp(0, c.shape[-1] - 1)
                return c.gather(-1, idx.unsqueeze(-1)).squeeze(-1)
            vals = self._ckpt(fn, logits[:, :, s:e])
            res = res + vals * in_chunk.to(logits.dtype)

        hard = lz - res
        if self.label_smoothing > 0:               # формула PyTorch: eps/V * sum(-log p_i), с градиентом
            uniform = lz - sx / V
            hard = (1 - self.label_smoothing) * hard + self.label_smoothing * uniform
        return hard.mean()

print("Реализации готовы")

## 3. Тест корректности: значения и градиенты против `F.cross_entropy`

Ожидаемо: **старый** лосс при `label_smoothing=0.1` расходится с эталоном (занижен, градиент сглаживающего слагаемого = 0), **новый** — совпадает до 1e-5.


In [ ]:
torch.manual_seed(0)
B, T, V = 4, 64, 32000
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))
eps = 0.1
ref = F.cross_entropy(logits.view(-1, V), targets.view(-1), label_smoothing=eps)

old = OldChunkedCE(chunk_size=1024, label_smoothing=eps)(logits, targets)
new = FixedChunkedCE(chunk_size=1024, label_smoothing=eps)(logits, targets)
print(f"Эталон F.cross_entropy : {ref.item():.6f}")
print(f"СТАРЫЙ chunked CE      : {old.item():.6f}  (расхождение {abs(old.item()-ref.item()):.6f})")
print(f"НОВЫЙ chunked CE       : {new.item():.6f}  (расхождение {abs(new.item()-ref.item()):.6f})")
print()
print("ВЕРДИКТ 1:", "СТАРЫЙ ЛОСС БЫЛ НЕВЕРЕН — ПОДТВЕРЖДЕНО" if abs(old.item()-ref.item()) > 1e-3 and abs(new.item()-ref.item()) < 1e-4 else "неожиданный результат")

## 4. Главный тест: VRAM при LoRA-обучении GPT-2

Реальный GPT-2 (124M) + LoRA r=16, seq=512. На каждом размере батча — 2 шага обучения, замер пика VRAM.
Ожидаемо: **новый** лосс идёт вровень со стандартным `F.cross_entropy`, **старый** — молча съедает
+2 байта на логит, и на большом батче ловит OOM там, где новый обучается спокойно.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Config
from peft import LoraConfig, get_peft_model

DEVICE = "cuda"
base = GPT2LMHeadModel.from_pretrained("gpt2")
model = get_peft_model(
    base.cuda(),
    LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
               target_modules=["c_attn", "c_proj"]),
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LoRA: {trainable/1e6:.2f}M обучаемых параметров из {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

In [ ]:
import gc

T = 512  # длина последовательности

def run_step(loss_fn, B, autocast_fp16=True):
    """Один шаг обучения: forward + backward + optimizer step. Возвращает peak VRAM в МБ или 'OOM'. """
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); gc.collect()
    try:
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
        ids = torch.randint(0, model.config.vocab_size, (B, T), device=DEVICE)
        ctx = torch.autocast("cuda", dtype=torch.float16) if autocast_fp16 else torch.enable_grad()
        with ctx:
            out = model(input_ids=ids)
            loss = loss_fn(out.logits, ids)
        loss.backward()
        opt.step(); opt.zero_grad(set_to_none=True)
        model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated() / 2**20
        del out, loss, ids, opt
        torch.cuda.reset_peak_memory_stats()
        return peak
    except torch.cuda.OutOfMemoryError:
        model.zero_grad(set_to_none=True)
        gc.collect(); torch.cuda.empty_cache()
        return None

std_ce   = lambda lg, t: F.cross_entropy(lg.view(-1, lg.shape[-1]).float(), t.view(-1), label_smoothing=0.1)
new_ce   = FixedChunkedCE(chunk_size=1024, label_smoothing=0.1)
old_ce   = OldChunkedCE(chunk_size=1024, label_smoothing=0.1)

names = [("старый chunked", old_ce), ("новый chunked", new_ce), ("F.cross_entropy", std_ce)]
results = {}
for B in (4, 8, 16, 32):
    row = {}
    for name, fn in names:
        row[name] = run_step(fn, B)
    results[B] = row
    cells = " | ".join(
        f"{name}: {('OOM' if v is None else f'{v:.0f} МБ')}" for name, v in row.items()
    )
    print(f"B={B:3} [{B*T*model.config.vocab_size*2/2**20:.0f} МБ logits fp16]  {cells}")

## 5. Вердикт



In [ ]:
print(f"{'Батч':>6} | {'старый':>12} | {'новый':>12} | {'F.cross_entropy':>15} | переплата старого")
print("-" * 72)
for B, row in results.items():
    o, n, s = row["старый chunked"], row["новый chunked"], row["F.cross_entropy"]
    def fmt(v): return "OOM" if v is None else f"{v:.0f} МБ"
    if o is None:
        extra = "OOM!"
    elif n is not None:
        extra = f"+{o - n:.0f} МБ ({(o/n-1)*100:+.0f}%)"
    else:
        extra = "?"
    print(f"{B:>6} | {fmt(o):>12} | {fmt(n):>12} | {fmt(s):>15} | {extra}")

o8, n8 = results[8]["старый chunked"], results[8]["новый chunked"]
if o8 and n8 and o8 > n8 * 1.15:
    print("\nИТОГ: баг ПОДТВЕРЖДЁН — старый ChunkedCE молча съедал сотни МБ VRAM,")
    print("а на больших батчах ловил OOM там, где исправленный обучается спокойно.")
    print("Это и была вероятная причина краша LoRA-обучения: 'память кончилась',")
    print("хотя по расчётам её должно было хватать.")
else:
    print("\nИТОГ: см. таблицу выше — сопоставь переплату старого лосса со свободной VRAM.")

## 6. Про «OOM, хотя памяти ещё было много»

Если OOM случался, когда `nvidia-smi` показывал свободные гигабайты — это **фрагментация**
кэширующего аллокатора PyTorch: память нарезана на куски, и большой запрос не помещается.
Лечится переменной окружения (поставить **до** первого обращения к CUDA):

```bash
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
```

Итоговые рекомендации для LoRA на 6–16 ГБ:
- обновить Selgis ML до версии с фиксом (новый ChunkedCE больше не ест лишнего);
- вместо увеличения батча — `gradient_accumulation_steps`;
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`;
- при желании `ce_chunk_size=512` (чуть меньше пик в backward).
